In [1]:
from pathlib import Path

from src.minbpe import RegexTokenizer
from src.gpt import GPTLanguageModel

import torch

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_dir = Path("data") / "tokenizer"
checkpoint_dir = Path("data") / "ch05_checkpoints"

In [3]:
tokenizer = RegexTokenizer()
tokenizer.load(model_file=str(tokenizer_dir / "tokenizer.model"))

#ckpt_files = sorted(
#    checkpoint_dir.glob("checkpoint_*.pt"),
#    key=lambda x: x.stat().st_ctime,
#    #key=lambda x: int(x.name.split("-")[1]),
#    reverse=True,
#)
#checkpoint_path = ckpt_files[0]

checkpoint_path = checkpoint_dir / "checkpoint_000054.pt"

print(f"load checkpoint: {checkpoint_path}")
checkpoint = torch.load(checkpoint_path, weights_only=True, map_location=device)
parameters = checkpoint['meta']['parameters']

load checkpoint: data/ch05_checkpoints/checkpoint_000054.pt


In [4]:
model = GPTLanguageModel(
    vocab_size=parameters['vocab_size'],
    block_size=parameters['block_size'],
    n_embd=parameters['n_embd'],
    n_head=parameters['n_head'],
    n_layer=parameters['n_layer'],
    dropout=parameters['dropout'],
    ignore_index=tokenizer.special_tokens["<|padding|>"],
    device=device,
)

model = torch.compile(model)
model.load_state_dict(checkpoint["model_state_dict"])

num_parameters = sum(p.numel() for p in model.parameters()) / 1e6
print(f'--> {num_parameters:_.3}M parameters')
# print_model_structure(model)

_ = model.eval()

--> 13.8M parameters


In [5]:
def into_tokens(role: str, content: str) -> torch.Tensor:
    d = tokenizer.special_tokens
    #print("~~~", d)

    input_msg = f"<|startoftext|>{role}<|separator|>{content}<|endoftext|>"
    print(f"--> {role} message: {input_msg}")

    input_tokens = tokenizer.encode(input_msg, allowed_special="all")

    return torch.tensor(input_tokens, dtype=torch.long).unsqueeze(0).to(device)


def ask_llm(input_tokens):
    model_answer = ""

    while True:
        output_tokens = model.generate(input_tokens=input_tokens, max_new_tokens=1)
        last_generated_token = output_tokens[0, -1].item()

        #print("~~~", input_tokens[0])
        model_answer += tokenizer.decode([last_generated_token])

        if last_generated_token == tokenizer.special_tokens["<|endoftext|>"]:
            break

        input_tokens = torch.cat((input_tokens, output_tokens[:, -1:]), dim=1)

        #if len(output_tokens[0]) > parameters['block_size']:
        #    break
        if len(input_tokens[0]) > parameters['block_size']:
            input_tokens = input_tokens[:, -parameters['block_size']:]

    return model_answer

In [6]:
user_content = "What's ChatGPT?"
input_tokens = into_tokens("user", user_content)

with torch.no_grad():
    output = model.generate(input_tokens=input_tokens, max_new_tokens=256)
    print("<-- assistant message:", tokenizer.decode(output[0].tolist()))

--> user message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|><|startoftext|>assistant<|separator|>ChatGPT is a large language model (LLM) developed by OpenAI, based on the LLM trained on a huge dataset of text data. It is open-sourced, and trained using a large dataset of text data from interface. The model itself consists of ruining time and awareness of patterns, i.e. show it to generate human-like responses to prompts.

Open Assistant is also open-sourced, meaning that it is made up of something that is not open-sourced and cannot publish or modify without any rules. This means that it can be made up of something like this one.

You should now be able to generate creative or expensive text inputs.<|endoftext|><|startoftext|>user<|separator|>Whow does the problem you want me to<|endoftext|><|startoftext|>assistant<|separator|>Naruts Do you have an imp


In [7]:
user_content = "What's ChatGPT?"
input_tokens = into_tokens("user", user_content)

answer = ask_llm(input_tokens)
print(f"<-- assistant message: {answer}")

--> user message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>assistant<|separator|>ChatGPT is a large language model research and architecture that is open source and accessible for this purpose, by definition, is also free and open source. 
ChatGPT is designed to be large and verified, with a large language model, resulting in a wide range of branches. 
For example, it can be used for datasets, models, self-attention analysis. 
It's recommended to put the text into parallelchemical scenarios.<|endoftext|>


In [8]:
user_content = "What's ChatGPT?"
input_tokens = into_tokens("user", user_content)

answer = ask_llm(input_tokens)
print(f"<-- assistant message: {answer}")

--> user message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>assistant<|separator|>ChatGPT is a large language model pretrained using vast amounts of data. Open Assistant is being created using a combination of a model that can be coded into text. The process of Open Assistant is published by an article from Google on a huggingface translation of each other and the associated spread of information.<|endoftext|>


In [9]:
user_content = "What's ChatGPT?"
input_tokens = into_tokens("user", user_content)

answer = ask_llm(input_tokens)
print(f"<-- assistant message: {answer}")

--> user message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>assistant<|separator|>ChatGPT is a large language model dedicated to building, run, and user input. Open Assistant is an open-source platform that is free to use. Open Assistant is also open source.io and there is also some companies such as Matrix, LAION, and LAION.<|endoftext|>
